In [15]:
#Imports pandas library to work with datasets.
import pandas as pd

In [16]:
#Installs openpyxl to enable reading Excels.
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [17]:
alvr_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_votereg_data_AlSoS.xlsx", sheet_name="October", header=None, engine="openpyxl")
rdh_og = pd.read_csv("../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv")
demographics_og = pd.read_csv("../data_storage/clean_data/demographics_2024_general.csv")

In [18]:
alvr = alvr_og.copy()
rdh = rdh_og.copy()
demographics = demographics_og.copy()

In [19]:
#RDH uses different column names so this renames and standardizes data
rdh = rdh.rename(columns={"countyname": "county_name", "countyfips": "fips_code"})
rdh["county_name"] = rdh["county_name"].str.strip().str.title()

#ALVR has a broken two-row header so manually set clean column names
alvr.columns = ['county_name','Total_AI','Active_Asian','Active_AmInd','Active_Black','Active_Fed','Active_Hispanic','Active_Korean','Active_White','Active_Other','Active_NotId','Active_Total','Inactive_Asian','Inactive_AmInd','Inactive_Black','Inactive_Fed','Inactive_Hispanic','Inactive_Korean','Inactive_White','Inactive_Other','Inactive_NotId','Inactive_Total']

#Drops rows 0 and 1 which contained the broken header labels then resets index
alvr = alvr.iloc[2:].reset_index(drop=True)
alvr["county_name"] = alvr["county_name"].astype(str).str.strip().str.title()

#Fill NaN in ALVR columns with 0 to handle missing values like Wilcox Inactive Hispanic
alvr_numeric_columns = ['Active_Asian','Active_AmInd','Active_Black','Active_Fed','Active_Hispanic',
    'Active_Korean','Active_White','Active_Other','Active_NotId','Active_Total',
    'Inactive_Asian','Inactive_AmInd','Inactive_Black','Inactive_Fed','Inactive_Hispanic',
    'Inactive_Korean','Inactive_White','Inactive_Other','Inactive_NotId','Inactive_Total']
alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)

/var/folders/_7/6hr00gjd7rgcmfz744n7rc2h0000gn/T/ipykernel_81157/2952026681.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)


In [20]:
#Filters the 17 counties for Alabama BB
BLACK_BELT_COUNTIES = ["Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas", "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike", "Bullock", "Macon", "Barbour", "Russell"]

alvr_bb = alvr[alvr["county_name"].isin(BLACK_BELT_COUNTIES)].copy()
rdh_bb = rdh[rdh["county_name"].isin(BLACK_BELT_COUNTIES)].copy()

In [21]:
#Inner join to merge ALVR with RDH for FIPS code and demographics for eligible population
registration = alvr_bb.merge(rdh_bb[["county_name", "fips_code"]], on="county_name")
registration = registration.merge(demographics[["county_name", "eligible_black", "eligible_white", "eligible_latinx"]], on="county_name")

In [22]:
#This calculates total registered by race (active + inactive)
registration["total_registered_black"] = registration["Active_Black"].astype(float) + registration["Inactive_Black"].astype(float)
registration["total_registered_white"] = registration["Active_White"].astype(float) + registration["Inactive_White"].astype(float)
registration["total_registered_latinx"] = registration["Active_Hispanic"].astype(float) + registration["Inactive_Hispanic"].astype(float)

In [23]:
#This calculates active registration rates: active divided by eligible population
#This can be used to show what share of eligible voters are actively registered and ready to vote
registration["reg_active_rate_black"] = (registration["Active_Black"].astype(float) / registration["eligible_black"]).round(4)
registration["reg_active_rate_white"] = (registration["Active_White"].astype(float) / registration["eligible_white"]).round(4)
registration["reg_active_rate_latinx"] = (registration["Active_Hispanic"].astype(float) / registration["eligible_latinx"]).round(4)

#This calculates the total registration rates: (active + inactive) divided by eligible population
#This can be used to show what share of eligible voters ever registered regardless of current status
#The Gap between active rate and total rate flags voters at risk of purge
registration["reg_total_rate_black"] = (registration["total_registered_black"] / registration["eligible_black"]).round(4)
registration["reg_total_rate_white"] = (registration["total_registered_white"] / registration["eligible_white"]).round(4)
registration["reg_total_rate_latinx"] = (registration["total_registered_latinx"] / registration["eligible_latinx"]).round(4)

In [24]:
#This calculates inactive rates to flag purge risk
#The overall inactive rate = total inactive divided by total registered
registration["inactive_rate"] = (registration["Inactive_Total"].astype(float) / (registration["Active_Total"].astype(float) + registration["Inactive_Total"].astype(float))).round(4)

#The inactive rate for each race
registration["inactive_rate_black"] = (registration["Inactive_Black"].astype(float) / registration["total_registered_black"]).round(4)
registration["inactive_rate_white"] = (registration["Inactive_White"].astype(float) / registration["total_registered_white"]).round(4)
registration["inactive_rate_latinx"] = (registration["Inactive_Hispanic"].astype(float) / registration["total_registered_latinx"]).round(4)

In [25]:
# Rename ALVR columns to follow naming convention: lowercase with underscores
registration = registration.rename(columns={
    # Active registered by race
    "Active_Black": "active_black",
    "Active_White": "active_white",
    "Active_Hispanic": "active_latinx",
    "Active_Total": "active_total",

    # Inactive registered by race
    "Inactive_Black": "inactive_black",
    "Inactive_White": "inactive_white",
    "Inactive_Hispanic": "inactive_latinx",
    "Inactive_Total": "inactive_total"
})

In [26]:
#Keep only final output columns in the order SPLC will read them
final_columns = [
    #Geographic identifiers
    "county_name", "fips_code",

    #Overall totals across all races
    "active_total", "inactive_total", "inactive_rate",

    #Black voters
    "eligible_black", "active_black", "inactive_black", "total_registered_black",
    "reg_active_rate_black", "reg_total_rate_black", "inactive_rate_black",

    #White voters
    "eligible_white", "active_white", "inactive_white", "total_registered_white",
    "reg_active_rate_white", "reg_total_rate_white", "inactive_rate_white",

    #Latinx voters
    "eligible_latinx", "active_latinx", "inactive_latinx", "total_registered_latinx",
    "reg_active_rate_latinx", "reg_total_rate_latinx", "inactive_rate_latinx"
]

registration = registration[final_columns].sort_values("county_name").reset_index(drop=True)

In [27]:
#Convert missing values to null per standardization document
registration = registration.fillna("null")

In [28]:
#Displays final cleaned registration dataset
registration
registration.to_csv("../data_storage/clean_data/voter_registration_2024_general.csv", index=False)